# If run on Colab:

In [ ]:
!pip install transformers sentence-transformers scikit-learn pandas numpy

# Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

In [6]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset


In [7]:
USE_SUBSET_DATA = False 
train_dataset = load_aokvqa(aokvqa_dir, 'train')  
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")





Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702


# Data Preparation

Fields Considered: 
- Question
- Choices
- Correct answer
- Correct Choice Indice
- rationale
- direct_answer

In [8]:

qa_data = []
for sample in train_dataset:
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)


In [9]:
qa_df.head()

,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer
0,What is the man by the bags awaiting?,"[skateboarder, train, delivery, cab]",cab,3,"A train would not be on the street, he would n...",ride ride bus taxi travelling traffic taxi cab...
1,Where does this man eat pizza?,"[office, cafe, motel, outside]",office,0,The man is eating pizza at a work desk in an o...,work office work work at work desk at desk off...
2,What is the occupation of the person driving?,"[waiter, farmer, cashier, musician]",farmer,1,The place is full of sheep that shows the pers...,farmer farmer bus driver farmer shepherd farme...
3,How were the drivers of the cars able to park ...,"[firemen, airport workers, police, postal work...",airport workers,1,These drivers work at the airport. Cars are pa...,airport workers driving stilts parking lot des...
4,How many people can ride this motorcycle at a ...,"[four, two, three, one]",two,1,Two people can be on the bike. There is a pass...,two two two two two two two two two two


# Unimodal Baselines (Text)

### Majority Class Baseline

In [10]:
from collections import Counter

# Count answer frequencies
all_answers = [sample['correct_answer'] for sample in qa_data]
most_common_answer = Counter(all_answers).most_common(1)[0][0]

# Predict using majority class
qa_df['prediction'] = most_common_answer
accuracy = (qa_df['prediction'] == qa_df['correct_answer']).mean()
print(f"Majority Class Baseline Accuracy: {accuracy:.2%}")


Majority Class Baseline Accuracy: 1.13%


### TF-IDF + Logistic Regression Baseline
- Format: `{question_str}`

In [11]:
print("Start vectorization fitting...")
vectorizer = TfidfVectorizer(max_features=5000)
X_text = vectorizer.fit_transform(qa_df['question'])
y = qa_df['correct_choice_idx']
X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.3, random_state=42)
print("Start model fitting...")
model_text_only = LogisticRegression(max_iter=1000, multi_class='multinomial')
model_text_only.fit(X_train, y_train)
y_pred_text = model_text_only.predict(X_test)
text_only_accuracy = accuracy_score(y_test, y_pred_text)
print(f"TF-IDF + Logistic Regression (All Rationales) Accuracy: {text_only_accuracy:.2%}")


Start vectorization fitting...
Start model fitting...
TF-IDF + Logistic Regression (All Rationales) Accuracy: 25.01%


### TF-IDF + Logistic Regression Baseline with Rationales
- Format: `{Question: question_str Rationale: r_1,...,r_n}`

In [12]:
print("Start vectorization fitting...")
qa_df['combined_input_direct'] = qa_df['question'] + " " + qa_df['rationale']
X_direct = vectorizer.fit_transform(qa_df['combined_input_direct'])
y_direct = qa_df['correct_choice_idx']
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_direct, y_direct, test_size=0.3, random_state=42)
print("Start model fitting...")
model_direct = LogisticRegression(max_iter=1000, multi_class='multinomial')
model_direct.fit(X_train_d, y_train_d)
y_pred_direct = model_direct.predict(X_test_d)
direct_answer_accuracy = accuracy_score(y_test_d, y_pred_direct)
print(f"TF-IDF + Logistic Regression (All Rationales) Accuracy: {direct_answer_accuracy:.2%}")



Start vectorization fitting...
Start model fitting...
TF-IDF + Logistic Regression (All Rationales) Accuracy: 24.17%


### BERT-Based Baseline (HuggingFace Transformers)

In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.empty_cache()

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

# Function to extract BERT embeddings on GPU
def get_bert_embeddings(texts):
    """
    Generate BERT embeddings for a list of texts on GPU.
    """
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # Move back to CPU for sklearn
    return embeddings


# Prepare data using combined question + rationales
qa_df['combined_input_rationale'] = qa_df['question'] + " " + qa_df['rationale']

# Extract BERT embeddings
print("Extracting BERT embeddings for questions + rationales...")
X_bert = get_bert_embeddings(qa_df['combined_input_rationale'].tolist())
y = qa_df['correct_choice_idx']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_bert, y, test_size=0.3, random_state=42)

# Train Logistic Regression on BERT embeddings
print("Training Logistic Regression with BERT embeddings...")
model_bert = LogisticRegression(max_iter=1000, multi_class='multinomial')
model_bert.fit(X_train, y_train)

# Predict and evaluate
y_pred_bert = model_bert.predict(X_test)
bert_accuracy = accuracy_score(y_test, y_pred_bert)
print(f"BERT + Logistic Regression Accuracy: {bert_accuracy:.2%}")


Extracting BERT embeddings for questions + rationales...


# Preprocessing Data
- Standardization